### Importing Libraries


In [149]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.neighbors import NearestNeighbors
from surprise import Dataset, Reader, SVD, KNNBasic
from surprise import accuracy
import joblib
from surprise import SVD

from collections import defaultdict
from surprise.model_selection import train_test_split, GridSearchCV
from surprise import accuracy
from pathlib import Path
# Load datasets


### reading csv files

In [150]:
movies = pd.read_csv(r"dataset\movies .csv")
ratings = pd.read_csv(r"dataset\ratings .csv")

# Display sample records
print("Movies Dataset:")
print(movies.head())

print("\nRatings Dataset:")
print(ratings.head())

# Dataset information
print("\nMovies Shape:", movies.shape)
print("Ratings Shape:", ratings.shape)

# Unique users and movies
print("\nTotal Users:", ratings["userId"].nunique())
print("Total Rated Movies:", ratings["movieId"].nunique())

Movies Dataset:
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

Ratings Dataset:
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931

Movies Shape: (9742, 3)
Ratings Shape: (100836, 4)

Total Users: 610
Total Rated Movies: 9724


### Creating the user-movie matrix


In [151]:
# Create the user-movie rating matrix

user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

# Display the first five users and movies
print(user_movie_matrix.head())

# Display matrix dimensions
print("\nRating Matrix Shape:", user_movie_matrix.shape)

# Count missing ratings
print("\nMissing Ratings:", user_movie_matrix.isna().sum().sum())

movieId  1       2       3       4       5       6       7       8       \
userId                                                                    
1           4.0     NaN     4.0     NaN     NaN     4.0     NaN     NaN   
2           NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
3           NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
4           NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
5           4.0     NaN     NaN     NaN     NaN     NaN     NaN     NaN   

movieId  9       10      ...  193565  193567  193571  193573  193579  193581  \
userId                   ...                                                   
1           NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN     NaN   
2           NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN     NaN   
3           NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN     NaN   
4           NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN     N

### Build item-based collaboration filtering

In [152]:

# Transpose the matrix so rows represent movies
movie_user_matrix = user_movie_matrix.T

# Fill missing ratings with zero for this baseline
movie_user_filled = movie_user_matrix.fillna(0)

# Calculate cosine similarity between movies
similarity_matrix = cosine_similarity(movie_user_filled)

print("Similarity Matrix Shape:", similarity_matrix.shape)

Similarity Matrix Shape: (9724, 9724)


### finding similar movies

In [153]:
# Create a DataFrame for movie similarities

movie_similarity = pd.DataFrame(
    similarity_matrix,
    index=movie_user_filled.index,
    columns=movie_user_filled.index
)

# Function to find similar movies
def get_similar_movies(movie_id, top_n=10):

    similarities = movie_similarity[movie_id]

    # Remove the selected movie itself
    similarities = similarities.drop(movie_id)

    # Get the most similar movies
    top_movies = similarities.sort_values(
        ascending=False
    ).head(top_n)

    # Get movie details
    result = movies[
        movies["movieId"].isin(top_movies.index)
    ].copy()

    # Add similarity scores
    result["similarity"] = result["movieId"].map(top_movies)

    return result.sort_values(
        by="similarity",
        ascending=False
    )

In [154]:
# Display the selected movie
print(movies[movies["movieId"] == 1])

# Recommend similar movies
recommendations = get_similar_movies(1, top_n=10)

print("\nRecommended Movies:")
print(recommendations[["title", "genres", "similarity"]])

   movieId             title                                       genres
0        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy

Recommended Movies:
                                                  title  \
2355                                 Toy Story 2 (1999)   
418                                Jurassic Park (1993)   
615                Independence Day (a.k.a. ID4) (1996)   
224           Star Wars: Episode IV - A New Hope (1977)   
314                                 Forrest Gump (1994)   
322                               Lion King, The (1994)   
911   Star Wars: Episode VI - Return of the Jedi (1983)   
546                          Mission: Impossible (1996)   
964                                Groundhog Day (1993)   
969                           Back to the Future (1985)   

                                               genres  similarity  
2355      Adventure|Animation|Children|Comedy|Fantasy    0.572601  
418                  Action|Adventure|Sci-Fi|

### search by movie title

In [155]:
def recommend_by_title(title, top_n=10):

    matched_movies = movies[
        movies["title"].str.contains(
            title,
            case=False,
            na=False,
            regex=False
        )
    ]

    if matched_movies.empty:
        print("Movie not found!")
        return

    # Select the first matching movie
    selected_movie = matched_movies.iloc[0]

    print("Selected Movie:", selected_movie["title"])

    return get_similar_movies(
        selected_movie["movieId"],
        top_n
    )

In [156]:
recommend_by_title("Toy Story", 10)

Selected Movie: Toy Story (1995)


,movieId,title,genres,similarity
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.572601
418,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,0.565637
615,780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller,0.564262
224,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,0.557388
314,356,Forrest Gump (1994),Comedy|Drama|Romance|War,0.547096
322,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,0.541145
911,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,0.541089
546,648,Mission: Impossible (1996),Action|Adventure|Mystery|Thriller,0.538913
964,1265,Groundhog Day (1993),Comedy|Fantasy|Romance,0.534169
969,1270,Back to the Future (1985),Adventure|Comedy|Sci-Fi,0.530381


In [157]:
recommend_by_title("Interstellar", 10)

Selected Movie: Interstellar (2014)


,movieId,title,genres,similarity
7372,79132,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,0.608150
8063,99114,Django Unchained (2012),Action|Drama|Western,0.587284
7768,91529,"Dark Knight Rises, The (2012)",Action|Adventure|Crime|IMAX,0.582879
8438,111759,Edge of Tomorrow (2014),Action|Sci-Fi|IMAX,0.570106
8569,116797,The Imitation Game (2014),Drama|Thriller|War,0.567939
7693,89745,"Avengers, The (2012)",Action|Adventure|Sci-Fi|IMAX,0.555597
8691,122904,Deadpool (2016),Action|Adventure|Comedy|Sci-Fi,0.554588
8879,134130,The Martian (2015),Adventure|Drama|Sci-Fi,0.546774
8475,112852,Guardians of the Galaxy (2014),Action|Adventure|Sci-Fi,0.539885
8372,109374,"Grand Budapest Hotel, The (2014)",Comedy|Drama,0.539152


In [158]:
recommend_by_title("Toy Story", 10)

Selected Movie: Toy Story (1995)


,movieId,title,genres,similarity
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.572601
418,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,0.565637
615,780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller,0.564262
224,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,0.557388
314,356,Forrest Gump (1994),Comedy|Drama|Romance|War,0.547096
322,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,0.541145
911,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,0.541089
546,648,Mission: Impossible (1996),Action|Adventure|Mystery|Thriller,0.538913
964,1265,Groundhog Day (1993),Comedy|Fantasy|Romance,0.534169
969,1270,Back to the Future (1985),Adventure|Comedy|Sci-Fi,0.530381


### Implement KNN based collaborative filtering

In [159]:

# Create the user-movie matrix
user_movie_filled = user_movie_matrix.fillna(0)

# Initialize the KNN model
knn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=6
)

# Train the model
knn_model.fit(user_movie_filled)

print("KNN model trained successfully!")

KNN model trained successfully!


### creating personalized recommendation function

In [160]:
def recommend_for_user(user_id, top_n=10):

    if user_id not in user_movie_filled.index:
        print("User ID not found!")
        return pd.DataFrame()

    # Get the selected user's ratings
    user_index = user_movie_filled.index.get_loc(user_id)

    user_vector = user_movie_filled.loc[user_id].values.reshape(1, -1)

    # Find similar users
    distances, indices = knn_model.kneighbors(
        user_vector,
        n_neighbors=6
    )

    # Convert cosine distances into similarities
    similarities = 1 - distances.flatten()

    # Collect recommendations from neighboring users
    scores = {}
    weights = {}

    for neighbor_index, similarity in zip(
        indices.flatten(),
        similarities
    ):
        neighbor_id = user_movie_filled.index[neighbor_index]

        # Exclude the target user
        if neighbor_id == user_id:
            continue

        neighbor_ratings = user_movie_matrix.loc[neighbor_id]

        for movie_id, rating in neighbor_ratings.items():

            # Recommend only movies the target user hasn't rated
            if (
                pd.isna(user_movie_matrix.loc[user_id, movie_id])
                and pd.notna(rating)
                and similarity > 0
            ):
                scores[movie_id] = (
                    scores.get(movie_id, 0)
                    + similarity * rating
                )

                weights[movie_id] = (
                    weights.get(movie_id, 0)
                    + similarity
                )

    # Calculate similarity-weighted scores
    predicted_scores = {
        movie_id: scores[movie_id] / weights[movie_id]
        for movie_id in scores
        if weights[movie_id] > 0
    }

    # Select the highest-scoring movies
    top_movies = sorted(
        predicted_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )[:top_n]

    # Retrieve movie details
    recommendations = movies[
        movies["movieId"].isin(
            [movie_id for movie_id, _ in top_movies]
        )
    ].copy()

    recommendations["predicted_rating"] = (
        recommendations["movieId"].map(predicted_scores)
    )

    return recommendations.sort_values(
        "predicted_rating",
        ascending=False
    ).reset_index(drop=True)

Testing

In [161]:
recommendations = recommend_for_user(user_id=1, top_n=10)

print(
    recommendations[
        ["title", "genres", "predicted_rating"]
    ]
)

                                               title  \
0                                Blade Runner (1982)   
1                                    Ref, The (1994)   
2  Dr. Strangelove or: How I Learned to Stop Worr...   
3                                     Stripes (1981)   
4                                  Roger & Me (1989)   
5                                  Dirty Work (1998)   
6                               Summer of Sam (1999)   
7                            Commitments, The (1991)   
8                            Fisher King, The (1991)   
9              Hard-Boiled (Lat sau san taam) (1992)   

                         genres  predicted_rating  
0        Action|Sci-Fi|Thriller               5.0  
1                        Comedy               5.0  
2                    Comedy|War               5.0  
3                    Comedy|War               5.0  
4                   Documentary               5.0  
5                        Comedy               5.0  
6                  

### SVD Implematation

In [162]:


# Define the rating scale
reader = Reader(rating_scale=(0.5, 5.0))

# Select the required columns
ratings_data = ratings[["userId", "movieId", "rating"]]

# Convert the Pandas DataFrame into Surprise format
data = Dataset.load_from_df(
    ratings_data,
    reader
)

print("Dataset prepared for SVD!")

Dataset prepared for SVD!


### traing the svd model

In [163]:
# Split the dataset into training and testing sets
trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

# Initialize SVD
svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    random_state=42
)

# Train the model
svd_model.fit(trainset)

print("SVD model trained successfully!")

SVD model trained successfully!


### evaluating the svd model

In [164]:
# Predict ratings for the test dataset
predictions = svd_model.test(testset)

# Calculate RMSE
rmse = accuracy.rmse(predictions)

# Calculate MAE
mae = accuracy.mae(predictions)

print("\nSVD Model Performance")
print("RMSE:", rmse)
print("MAE:", mae)

RMSE: 0.8807
MAE:  0.6766

SVD Model Performance
RMSE: 0.8807462819979623
MAE: 0.6765729095860605


### generate personalized recommendation using svd

In [165]:
def recommend_with_svd(user_id, top_n=10):

    if user_id not in ratings["userId"].unique():
        print("User ID not found!")
        return pd.DataFrame()

    # Find movies the user has already rated
    rated_movies = set(
        ratings.loc[
            ratings["userId"] == user_id,
            "movieId"
        ]
    )

    # Consider movies present in the model's training data
    all_movie_ids = set(trainset.all_items())

    # Convert internal movie indices to original movie IDs
    all_movie_ids = {
        trainset.to_raw_iid(inner_id)
        for inner_id in all_movie_ids
    }

    # Identify unseen movies
    unseen_movies = [
        movie_id
        for movie_id in all_movie_ids
        if movie_id not in rated_movies
    ]

    # Predict ratings for unseen movies
    predictions = [
        (
            movie_id,
            svd_model.predict(
                str(user_id),
                str(movie_id)
            ).est
        )
        for movie_id in unseen_movies
    ]

    # Select the top-rated predictions
    top_predictions = sorted(
        predictions,
        key=lambda x: x[1],
        reverse=True
    )[:top_n]

    # Create the recommendation DataFrame
    result = pd.DataFrame(
        top_predictions,
        columns=["movieId", "predicted_rating"]
    )

    result = result.merge(
        movies,
        on="movieId",
        how="left"
    )

    return result[[
        "title",
        "genres",
        "predicted_rating"
    ]]

### testing it

In [166]:
recommendations = recommend_with_svd(
    user_id=1,
    top_n=10
)

print(recommendations)

                                title                        genres  \
0                      Jumanji (1995)    Adventure|Children|Fantasy   
1            Waiting to Exhale (1995)          Comedy|Drama|Romance   
2  Father of the Bride Part II (1995)                        Comedy   
3                      Sabrina (1995)                Comedy|Romance   
4                 Tom and Huck (1995)            Adventure|Children   
5                 Sudden Death (1995)                        Action   
6                    GoldenEye (1995)     Action|Adventure|Thriller   
7      American President, The (1995)          Comedy|Drama|Romance   
8  Dracula: Dead and Loving It (1995)                 Comedy|Horror   
9                        Balto (1995)  Adventure|Animation|Children   

   predicted_rating  
0          3.503229  
1          3.503229  
2          3.503229  
3          3.503229  
4          3.503229  
5          3.503229  
6          3.503229  
7          3.503229  
8          3.503229 

### Loading your existing dataset


In [167]:

BASE_DIR = Path.cwd()

if not (BASE_DIR / "dataset").exists():
    BASE_DIR = BASE_DIR.parent

movies = pd.read_csv(BASE_DIR / "dataset" / "movies .csv")
ratings = pd.read_csv(BASE_DIR / "dataset" / "ratings .csv")

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)

display(movies.head())
display(ratings.head())

Movies: (9742, 3)
Ratings: (100836, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [168]:
print("Missing values:")
print(ratings.isnull().sum())

print("Duplicate ratings:", ratings.duplicated(
    subset=["userId", "movieId"]
).sum())

print("Rating range:")
print(ratings["rating"].min(), ratings["rating"].max())

Missing values:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
Duplicate ratings: 0
Rating range:
0.5 5.0


Prepare the data for ml

In [169]:
reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

print("Data prepared successfully!")



Data prepared successfully!


### creatinng a baseline svd

In [170]:
trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

baseline_model = SVD(
    n_factors=100,
    n_epochs=20,
    random_state=42
)

baseline_model.fit(trainset)

baseline_predictions = baseline_model.test(testset)

baseline_rmse = accuracy.rmse(
    baseline_predictions,
    verbose=True
)

baseline_mae = accuracy.mae(
    baseline_predictions,
    verbose=True
)

print("Baseline RMSE:", baseline_rmse)
print("Baseline MAE:", baseline_mae)

RMSE: 0.8807
MAE:  0.6766
Baseline RMSE: 0.8807462819979623
Baseline MAE: 0.6765729095860605


### Improve SVD using hyperparameter tuning

In [171]:
param_grid = {
    "n_factors": [50, 100, 150],
    "n_epochs": [20, 30],
    "lr_all": [0.002, 0.005],
    "reg_all": [0.02, 0.1]
}

grid_search = GridSearchCV(
    SVD,
    param_grid,
    measures=["rmse", "mae"],
    cv=3,
    joblib_verbose=1
)

grid_search.fit(data)

print("Best RMSE:", grid_search.best_score["rmse"])
print("Best RMSE parameters:", grid_search.best_params["rmse"])

print("Best MAE:", grid_search.best_score["mae"])
print("Best MAE parameters:", grid_search.best_params["mae"])


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:  1.8min


Best RMSE: 0.870453024936749
Best RMSE parameters: {'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.1}
Best MAE: 0.6697152102727401
Best MAE parameters: {'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.1}


[Parallel(n_jobs=1)]: Done  72 out of  72 | elapsed:  2.7min finished


### Evaluate the improved model

In [172]:
best_params = grid_search.best_params["rmse"]

improved_model = SVD(
    **best_params,
    random_state=42
)

improved_model.fit(trainset)

improved_predictions = improved_model.test(testset)

improved_rmse = accuracy.rmse(
    improved_predictions,
    verbose=True
)

improved_mae = accuracy.mae(
    improved_predictions,
    verbose=True
)

RMSE: 0.8718
MAE:  0.6700


### comparing baseline svd and tuned svd

In [173]:
comparison = pd.DataFrame({
    "Model": ["Baseline SVD", "Tuned SVD"],
    "RMSE": [baseline_rmse, improved_rmse],
    "MAE": [baseline_mae, improved_mae]
})

display(comparison)

,Model,RMSE,MAE
0,Baseline SVD,0.880746,0.676573
1,Tuned SVD,0.871757,0.669961


### saving the best model

In [174]:
joblib.dump(
    improved_model,
    BASE_DIR / "svd_model.pkl"
)

print("Model saved successfully!")

Model saved successfully!


In [175]:
BASE_DIR = Path.cwd()

if not (BASE_DIR / "dataset").exists():
    BASE_DIR = BASE_DIR.parent

movies = pd.read_csv(BASE_DIR / "dataset" / "movies .csv")
ratings = pd.read_csv(BASE_DIR / "dataset" / "ratings .csv")

print("Total movies:", len(movies))
print("Total ratings:", len(ratings))
print("Unique users:", ratings["userId"].nunique())

Total movies: 9742
Total ratings: 100836
Unique users: 610


In [176]:
reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    random_state=42
)

svd_model.fit(trainset)

predictions = svd_model.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

print("\nBaseline SVD Results")
print("RMSE:", round(rmse, 4))
print("MAE:", round(mae, 4))

RMSE: 0.8807
MAE:  0.6766

Baseline SVD Results
RMSE: 0.8807
MAE: 0.6766


In [177]:

param_grid = {
    "n_factors": [50, 100, 150],
    "n_epochs": [20, 30],
    "lr_all": [0.002, 0.005],
    "reg_all": [0.02, 0.1]
}

grid_search = GridSearchCV(
    SVD,
    param_grid,
    measures=["rmse", "mae"],
    cv=3,
    joblib_verbose=1
)

grid_search.fit(data)

print("Best RMSE:", grid_search.best_score["rmse"])
print("Best RMSE parameters:", grid_search.best_params["rmse"])

print("Best MAE:", grid_search.best_score["mae"])
print("Best MAE parameters:", grid_search.best_params["mae"])

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:  1.3min


Best RMSE: 0.8705925366180415
Best RMSE parameters: {'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.1}
Best MAE: 0.669445528680012
Best MAE parameters: {'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.1}


[Parallel(n_jobs=1)]: Done  72 out of  72 | elapsed:  2.0min finished


In [178]:
best_params = grid_search.best_params["rmse"]

improved_model = SVD(
    **best_params,
    random_state=42
)

improved_model.fit(trainset)

improved_predictions = improved_model.test(testset)

improved_rmse = accuracy.rmse(
    improved_predictions,
    verbose=True
)

improved_mae = accuracy.mae(
    improved_predictions,
    verbose=True
)

print("\nImproved SVD Results")
print("RMSE:", round(improved_rmse, 4))
print("MAE:", round(improved_mae, 4))

RMSE: 0.8723
MAE:  0.6701

Improved SVD Results
RMSE: 0.8723
MAE: 0.6701


In [179]:
sim_options = {
    "name": "cosine",
    "user_based": True
}

knn_model = KNNBasic(
    k=40,
    min_k=1,
    sim_options=sim_options,
    verbose=False
)

knn_model.fit(trainset)

knn_predictions = knn_model.test(testset)

knn_rmse = accuracy.rmse(
    knn_predictions,
    verbose=True
)

knn_mae = accuracy.mae(
    knn_predictions,
    verbose=True
)

print("\nUser-Based KNN Results")
print("RMSE:", round(knn_rmse, 4))
print("MAE:", round(knn_mae, 4))

RMSE: 0.9823
MAE:  0.7559

User-Based KNN Results
RMSE: 0.9823
MAE: 0.7559


In [180]:


comparison = pd.DataFrame({
    "Model": ["Baseline SVD", "Improved SVD", "User-Based KNN"],
    "RMSE": [rmse, improved_rmse, knn_rmse],
    "MAE": [mae, improved_mae, knn_mae]
})

display(comparison)

,Model,RMSE,MAE
0,Baseline SVD,0.880746,0.676573
1,Improved SVD,0.872317,0.670102
2,User-Based KNN,0.982256,0.755897


In [181]:
sim_options = {
    "name": "cosine",
    "user_based": False
}

item_knn_model = KNNBasic(
    k=40,
    min_k=1,
    sim_options=sim_options,
    verbose=False
)

item_knn_model.fit(trainset)

item_knn_predictions = item_knn_model.test(testset)

item_knn_rmse = accuracy.rmse(
    item_knn_predictions,
    verbose=True
)

item_knn_mae = accuracy.mae(
    item_knn_predictions,
    verbose=True
)

print("\nItem-Based KNN Results")
print("RMSE:", round(item_knn_rmse, 4))
print("MAE:", round(item_knn_mae, 4))

RMSE: 0.9800
MAE:  0.7615

Item-Based KNN Results
RMSE: 0.98
MAE: 0.7615


In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Baseline SVD",
        "Improved SVD",
        "User-Based KNN",
        "Item-Based KNN"
    ],
    "RMSE": [
        rmse,
        improved_rmse,
        knn_rmse,
        item_knn_rmse
    ],
    "MAE": [
        mae,
        improved_mae,
        knn_mae,
        item_knn_mae
    ]
})

display(comparison.sort_values("RMSE"))

,Model,RMSE,MAE
1,Improved SVD,0.872317,0.670102
0,Baseline SVD,0.880746,0.676573
3,Item-Based KNN,0.979957,0.761470
2,User-Based KNN,0.982256,0.755897


In [183]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies["genres_clean"] = movies["genres"].str.replace("|", " ", regex=False)

tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(movies["genres_clean"])

genre_similarity = cosine_similarity(genre_matrix)

print("Genre similarity matrix:", genre_similarity.shape)

Genre similarity matrix: (9742, 9742)


In [184]:
def recommend_hybrid(user_id, favorite_movie_ids, top_n=10,
                     svd_weight=0.7, genre_weight=0.3):

    rated_movies = set(
        ratings.loc[ratings["userId"] == user_id, "movieId"]
    )

    favorite_indices = movies.index[
        movies["movieId"].isin(favorite_movie_ids)
    ].tolist()

    if not favorite_indices:
        raise ValueError("No favorite movie IDs found in the movies dataset.")

    genre_scores = genre_similarity[favorite_indices].mean(axis=0)

    candidates = movies.loc[
        ~movies["movieId"].isin(rated_movies)
    ].copy()

    candidates["svd_score"] = candidates["movieId"].apply(
        lambda movie_id: improved_model.predict(
            user_id, int(movie_id)
        ).est
    )

    candidates["genre_score"] = candidates.index.map(
        lambda index: genre_scores[index]
    )

    rating_min = candidates["svd_score"].min()
    rating_max = candidates["svd_score"].max()

    if rating_max > rating_min:
        candidates["svd_normalized"] = (
            (candidates["svd_score"] - rating_min)
            / (rating_max - rating_min)
        )
    else:
        candidates["svd_normalized"] = 0.5

    candidates["hybrid_score"] = (
        svd_weight * candidates["svd_normalized"]
        + genre_weight * candidates["genre_score"]
    )

    return candidates.sort_values(
        "hybrid_score", ascending=False
    )[["movieId", "title", "genres", "svd_score",
       "genre_score", "hybrid_score"]].head(top_n)

In [185]:
movies[["movieId", "title", "genres"]].head(10)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [186]:
recommend_hybrid(
    user_id=1,
    favorite_movie_ids=[1, 260],
    top_n=10
)

,movieId,title,genres,svd_score,genre_score,hybrid_score
924,1223,"Grand Day Out with Wallace and Gromit, A (1989)",Adventure|Animation|Children|Comedy|Sci-Fi,4.919490,0.670894,0.875676
596,741,Ghost in the Shell (Kôkaku kidôtai) (1995),Animation|Sci-Fi,4.992928,0.455791,0.834489
585,720,Wallace & Gromit: The Best of Aardman Animatio...,Adventure|Animation|Comedy,4.884235,0.501261,0.813579
4348,6350,Laputa: Castle in the Sky (Tenkû no shiro Rapy...,Action|Adventure|Animation|Children|Fantasy|Sc...,4.660710,0.731661,0.811645
8301,106642,"Day of the Doctor, The (2013)",Adventure|Drama|Sci-Fi,4.795653,0.537149,0.796187
4800,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,4.828941,0.496731,0.794643
5954,34405,Serenity (2005),Action|Adventure|Sci-Fi,4.714670,0.602695,0.790108
3984,5618,Spirited Away (Sen to Chihiro no kamikakushi) ...,Adventure|Animation|Fantasy,4.777961,0.535477,0.790062
2283,3030,Yojimbo (1961),Action|Adventure,4.824949,0.484514,0.789709
6772,60069,WALL·E (2008),Adventure|Animation|Children|Romance|Sci-Fi,4.681920,0.618057,0.784307


In [187]:
favorite_movie_ids=[1, 260]

In [188]:
weight_results = []

for svd_weight in [0.5, 0.6, 0.7, 0.8, 0.9]:
    genre_weight = 1 - svd_weight

    recommendations = recommend_hybrid(
        user_id=1,
        favorite_movie_ids=[1, 260],
        top_n=10,
        svd_weight=svd_weight,
        genre_weight=genre_weight
    )

    weight_results.append({
        "SVD Weight": svd_weight,
        "Genre Weight": genre_weight,
        "Top Recommendation": recommendations.iloc[0]["title"],
        "Average Hybrid Score": recommendations["hybrid_score"].mean()
    })

weight_results = pd.DataFrame(weight_results)

display(weight_results)

,SVD Weight,Genre Weight,Top Recommendation,Average Hybrid Score
0,0.5,0.5,"Grand Day Out with Wallace and Gromit, A (1989)",0.742623
1,0.6,0.4,"Grand Day Out with Wallace and Gromit, A (1989)",0.773706
2,0.7,0.3,"Grand Day Out with Wallace and Gromit, A (1989)",0.808041
3,0.8,0.2,"Grand Day Out with Wallace and Gromit, A (1989)",0.847409
4,0.9,0.1,Ghost in the Shell (Kôkaku kidôtai) (1995),0.901432


In [189]:


def evaluate_hybrid(
    predictions,
    ratings,
    movies,
    genre_similarity,
    svd_weight=0.7,
    genre_weight=0.3,
    k=10
):
    test_df = pd.DataFrame([
        {
            "userId": int(pred.uid),
            "movieId": int(pred.iid),
            "actual": float(pred.r_ui),
            "svd_score": float(pred.est)
        }
        for pred in predictions
    ])

    movie_indices = pd.Series(
        movies.index,
        index=movies["movieId"]
    ).to_dict()

    results = []

    for user_id, user_test in test_df.groupby("userId"):
        history = ratings[
            (ratings["userId"] == user_id) &
            (ratings["rating"] >= 4.0)
        ]

        favorite_ids = history["movieId"].tolist()

        favorite_indices = [
            movie_indices[mid]
            for mid in favorite_ids
            if mid in movie_indices
        ]

        if not favorite_indices:
            continue

        genre_scores = genre_similarity[favorite_indices].mean(axis=0)

        user_test = user_test[
            user_test["movieId"].isin(movie_indices)
        ].copy()

        if user_test.empty:
            continue

        user_test["genre_score"] = user_test["movieId"].map(
            lambda mid: genre_scores[movie_indices[mid]]
        )

        minimum = user_test["svd_score"].min()
        maximum = user_test["svd_score"].max()

        if maximum > minimum:
            user_test["svd_normalized"] = (
                (user_test["svd_score"] - minimum) /
                (maximum - minimum)
            )
        else:
            user_test["svd_normalized"] = 0.5

        user_test["hybrid_score"] = (
            svd_weight * user_test["svd_normalized"] +
            genre_weight * user_test["genre_score"]
        )

        relevant = set(
            user_test.loc[
                user_test["actual"] >= 4.0, "movieId"
            ]
        )

        if not relevant:
            continue

        top_k = user_test.nlargest(k, "hybrid_score")
        recommended = list(top_k["movieId"])

        hits = sum(mid in relevant for mid in recommended)

        precision = hits / k
        recall = hits / len(relevant)

        discounts = np.log2(np.arange(2, len(top_k) + 2))
        gains = np.array([
            1.0 if mid in relevant else 0.0
            for mid in recommended
        ])

        dcg = np.sum(gains / discounts)

        ideal_gains = np.ones(min(len(relevant), k))
        ideal_discounts = np.log2(
            np.arange(2, len(ideal_gains) + 2)
        )
        idcg = np.sum(ideal_gains / ideal_discounts)

        ndcg = dcg / idcg if idcg > 0 else 0.0

        results.append({
            "userId": user_id,
            "Precision@10": precision,
            "Recall@10": recall,
            "NDCG@10": ndcg
        })

    return pd.DataFrame(results)

In [190]:


train_ratings = pd.DataFrame(
    trainset.all_ratings(),
    columns=["inner_user", "inner_movie", "rating"]
)

train_ratings["userId"] = train_ratings["inner_user"].map(
    lambda x: int(trainset.to_raw_uid(x))
)

train_ratings["movieId"] = train_ratings["inner_movie"].map(
    lambda x: int(trainset.to_raw_iid(x))
)

train_ratings = train_ratings[
    ["userId", "movieId", "rating"]
]

movie_id_to_index = {
    int(movie_id): index
    for index, movie_id in enumerate(movies["movieId"])
}

print("Training ratings:", len(train_ratings))
print("Users:", train_ratings["userId"].nunique())
print("Movies:", len(movie_id_to_index))

Training ratings: 80668
Users: 610
Movies: 9742


In [191]:
def evaluate_recommenders(
    predictions,
    train_ratings,
    genre_similarity,
    movies,
    k=10,
    svd_weight=0.7,
    genre_weight=0.3
):
    movie_id_to_index = {
        int(movie_id): index
        for index, movie_id in enumerate(movies["movieId"])
    }

    train_seen = (
        train_ratings.groupby("userId")["movieId"]
        .apply(set)
        .to_dict()
    )

    train_favorites = (
        train_ratings[train_ratings["rating"] >= 4.0]
        .groupby("userId")["movieId"]
        .apply(list)
        .to_dict()
    )

    test_df = pd.DataFrame([
        {
            "userId": int(pred.uid),
            "movieId": int(pred.iid),
            "actual": float(pred.r_ui),
            "svd_score": float(pred.est)
        }
        for pred in predictions
    ])

    all_results = []

    for user_id, group in test_df.groupby("userId"):
        favorites = train_favorites.get(user_id, [])

        favorite_indices = [
            movie_id_to_index[mid]
            for mid in favorites
            if mid in movie_id_to_index
        ]

        if not favorite_indices:
            continue

        genre_scores = genre_similarity[favorite_indices].mean(axis=0)

        candidates = group[
            ~group["movieId"].isin(train_seen.get(user_id, set()))
        ].copy()

        if candidates.empty:
            continue

        candidates["genre_score"] = candidates["movieId"].map(
            lambda mid: genre_scores[movie_id_to_index[mid]]
        )

        minimum = candidates["svd_score"].min()
        maximum = candidates["svd_score"].max()

        if maximum > minimum:
            candidates["svd_normalized"] = (
                (candidates["svd_score"] - minimum)
                / (maximum - minimum)
            )
        else:
            candidates["svd_normalized"] = 0.5

        candidates["hybrid_score"] = (
            svd_weight * candidates["svd_normalized"]
            + genre_weight * candidates["genre_score"]
        )

        relevant = set(
            candidates.loc[
                candidates["actual"] >= 4.0, "movieId"
            ]
        )

        if not relevant:
            continue

        for model_name, score_column in [
            ("SVD", "svd_score"),
            ("Hybrid", "hybrid_score")
        ]:
            ranked = candidates.sort_values(
                score_column, ascending=False
            ).head(k)

            recommended = ranked["movieId"].tolist()

            hits = sum(
                movie_id in relevant
                for movie_id in recommended
            )

            precision = hits / k
            recall = hits / len(relevant)

            discounts = np.log2(
                np.arange(2, len(recommended) + 2)
            )

            gains = np.array([
                1.0 if movie_id in relevant else 0.0
                for movie_id in recommended
            ])

            dcg = np.sum(gains / discounts)

            ideal_length = min(len(relevant), k)

            ideal_gains = np.ones(ideal_length)
            ideal_discounts = np.log2(
                np.arange(2, ideal_length + 2)
            )

            idcg = np.sum(ideal_gains / ideal_discounts)

            ndcg = dcg / idcg if idcg > 0 else 0.0

            all_results.append({
                "userId": user_id,
                "Model": model_name,
                "Precision@10": precision,
                "Recall@10": recall,
                "NDCG@10": ndcg
            })

    results = pd.DataFrame(all_results)

    if results.empty:
        raise ValueError(
            "No eligible users found. Check your training history "
            "and test predictions."
        )

    return results

In [192]:
evaluation_results = evaluate_recommenders(
    predictions=predictions,
    train_ratings=train_ratings,
    genre_similarity=genre_similarity,
    movies=movies,
    k=10,
    svd_weight=0.7,
    genre_weight=0.3
)

summary = (
    evaluation_results
    .groupby("Model")[
        ["Precision@10", "Recall@10", "NDCG@10"]
    ]
    .mean()
    .round(4)
)

display(summary)

,Precision@10,Recall@10,NDCG@10
Model,,,
Hybrid,0.5736,0.6778,0.8055
SVD,0.5714,0.6776,0.8019


In [193]:
improved_predictions = improved_model.test(testset)

improved_evaluation = evaluate_recommenders(
    predictions=improved_predictions,
    train_ratings=train_ratings,
    genre_similarity=genre_similarity,
    movies=movies,
    k=10,
    svd_weight=0.7,
    genre_weight=0.3
)

display(
    improved_evaluation
    .groupby("Model")[
        ["Precision@10", "Recall@10", "NDCG@10"]
    ]
    .mean()
    .round(4)
)

,Precision@10,Recall@10,NDCG@10
Model,,,
Hybrid,0.5782,0.6815,0.8050
SVD,0.5729,0.6774,0.7983


In [194]:
svd_weight = 0.7
genre_weight = 0.3

In [195]:
weight_results = []

for svd_weight in [0.5, 0.6, 0.7, 0.8, 0.9]:
    genre_weight = 1 - svd_weight

    result = evaluate_recommenders(
        predictions=improved_predictions,
        train_ratings=train_ratings,
        genre_similarity=genre_similarity,
        movies=movies,
        k=10,
        svd_weight=svd_weight,
        genre_weight=genre_weight
    )

    hybrid_result = result[result["Model"] == "Hybrid"]

    weight_results.append({
        "SVD Weight": svd_weight,
        "Genre Weight": genre_weight,
        "Precision@10": hybrid_result["Precision@10"].mean(),
        "Recall@10": hybrid_result["Recall@10"].mean(),
        "NDCG@10": hybrid_result["NDCG@10"].mean()
    })

weight_comparison = pd.DataFrame(weight_results)

display(
    weight_comparison.sort_values(
        "NDCG@10", ascending=False
    ).round(4)
)

,SVD Weight,Genre Weight,Precision@10,Recall@10,NDCG@10
0,0.5,0.5,0.5756,0.6808,0.8065
3,0.8,0.2,0.5785,0.6821,0.8055
2,0.7,0.3,0.5782,0.6815,0.8050
1,0.6,0.4,0.5756,0.6814,0.8045
4,0.9,0.1,0.5756,0.6800,0.8027


In [196]:


# 1. Train the tuned SVD model
svd_model = SVD(
    n_factors=100,
    n_epochs=30,
    lr_all=0.005,
    reg_all=0.1,
    random_state=42
)

svd_model.fit(trainset)

# 2. Train User-based KNN
user_knn = KNNBasic(
    sim_options={
        "name": "cosine",
        "user_based": True
    },
    verbose=False
)

user_knn.fit(trainset)

# 3. Train Item-based KNN
item_knn = KNNBasic(
    sim_options={
        "name": "cosine",
        "user_based": False
    },
    verbose=False
)

item_knn.fit(trainset)

print("All three models trained successfully!")

All three models trained successfully!


In [197]:


models = {
    "SVD": svd_model,
    "User-based KNN": user_knn,
    "Item-based KNN": item_knn
}

rating_results = []

for model_name, model in models.items():

    predictions = model.test(testset)

    rmse = accuracy.rmse(predictions, verbose=False)
    mae = accuracy.mae(predictions, verbose=False)

    rating_results.append({
        "Model": model_name,
        "RMSE": rmse,
        "MAE": mae
    })

rating_results_df = pd.DataFrame(rating_results)

rating_results_df = rating_results_df.sort_values(
    by="RMSE"
).reset_index(drop=True)

rating_results_df

,Model,RMSE,MAE
0,SVD,0.871757,0.669961
1,Item-based KNN,0.979957,0.761470
2,User-based KNN,0.982256,0.755897


In [198]:


K = 10
RELEVANCE_THRESHOLD = 4.0


def evaluate_ranking(model, testset, trainset, k=10):

    # Collect held-out ratings and model predictions by user
    user_predictions = defaultdict(list)

    for user_id, movie_id, actual_rating in testset:

        prediction = model.predict(user_id, movie_id)

        user_predictions[user_id].append({
            "movie_id": movie_id,
            "actual_rating": actual_rating,
            "predicted_rating": prediction.est
        })

    # Movies each user already rated during training
    train_history = defaultdict(set)

    for inner_user_id, user_ratings in trainset.ur.items():

        raw_user_id = trainset.to_raw_uid(inner_user_id)

        for inner_movie_id, rating in user_ratings:
            raw_movie_id = trainset.to_raw_iid(inner_movie_id)
            train_history[raw_user_id].add(raw_movie_id)

    precisions = []
    recalls = []
    ndcgs = []

    for user_id, items in user_predictions.items():

        # Remove movies already present in the user's training history
        items = [
            item for item in items
            if item["movie_id"] not in train_history[user_id]
        ]

        if not items:
            continue

        # Relevant movies according to their actual test ratings
        relevant_movies = {
            item["movie_id"]
            for item in items
            if item["actual_rating"] >= RELEVANCE_THRESHOLD
        }

        # Rank movies by predicted rating
        ranked_items = sorted(
            items,
            key=lambda x: x["predicted_rating"],
            reverse=True
        )

        top_items = ranked_items[:k]

        # Precision@K
        hits = sum(
            item["movie_id"] in relevant_movies
            for item in top_items
        )

        precision = hits / k

        # Recall@K
        recall = (
            hits / len(relevant_movies)
            if relevant_movies
            else 0.0
        )

        # NDCG@K
        dcg = sum(
            1 / np.log2(rank + 2)
            for rank, item in enumerate(top_items)
            if item["movie_id"] in relevant_movies
        )

        ideal_hits = min(len(relevant_movies), k)

        idcg = sum(
            1 / np.log2(rank + 2)
            for rank in range(ideal_hits)
        )

        ndcg = dcg / idcg if idcg > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        ndcgs.append(ndcg)

    return {
        "Precision@10": np.mean(precisions) if precisions else 0.0,
        "Recall@10": np.mean(recalls) if recalls else 0.0,
        "NDCG@10": np.mean(ndcgs) if ndcgs else 0.0,
        "Evaluated users": len(precisions)
    }

In [199]:
ranking_results = []

for model_name, model in models.items():

    metrics = evaluate_ranking(
        model=model,
        testset=testset,
        trainset=trainset,
        k=10
    )

    ranking_results.append({
        "Model": model_name,
        **metrics
    })

ranking_results_df = pd.DataFrame(ranking_results)

ranking_results_df = ranking_results_df.sort_values(
    by="NDCG@10",
    ascending=False
).reset_index(drop=True)

ranking_results_df

,Model,Precision@10,Recall@10,NDCG@10,Evaluated users
0,SVD,0.563934,0.666826,0.786191,610
1,User-based KNN,0.560820,0.663742,0.777392,610
2,Item-based KNN,0.490984,0.629267,0.684223,610


In [200]:
user_id = 1
movie_id = 1

prediction = svd_model.predict(user_id, movie_id)

print("Predicted rating:", prediction.est)

Predicted rating: 4.557530564390438


In [201]:
import numpy as np

def estimate_new_user_preferences(
    model,
    user_ratings,
    epochs=100,
    learning_rate=0.01,
    reg=0.1
):
    n_factors = model.qi.shape[1]

    user_factors = np.zeros(n_factors)
    user_bias = 0.0

    global_mean = model.trainset.global_mean

    rated_movies = []

    for movie_id, rating in user_ratings.items():
        try:
            inner_movie_id = model.trainset.to_inner_iid(movie_id)
            movie_factors = model.qi[inner_movie_id]
            movie_bias = model.bi[inner_movie_id]

            rated_movies.append(
                (rating, movie_factors, movie_bias)
            )

        except ValueError:
            print(f"Movie ID {movie_id} not found in the trained model.")

    if not rated_movies:
        raise ValueError(
            "None of the supplied movie IDs exist in the trained model."
        )

    for _ in range(epochs):
        for rating, movie_factors, movie_bias in rated_movies:

            prediction = (
                global_mean
                + user_bias
                + movie_bias
                + np.dot(user_factors, movie_factors)
            )

            error = rating - prediction

            old_user_factors = user_factors.copy()
            old_user_bias = user_bias

            user_factors += learning_rate * (
                error * movie_factors
                - reg * old_user_factors
            )

            user_bias += learning_rate * (
                error - reg * old_user_bias
            )

    return user_factors, user_bias

In [202]:
print("estimate_new_user_preferences:",
      "estimate_new_user_preferences" in globals())

print("estimate_new_user_profile:",
      "estimate_new_user_profile" in globals())

estimate_new_user_preferences: True
estimate_new_user_profile: True


In [203]:
my_ratings = {
    1: 5.0,
    50: 4.5,
    260: 5.0
}

my_user_factors, my_user_bias = estimate_new_user_preferences(
    svd_model,
    my_ratings
)

print("Personalized user profile created!")
print("Number of latent factors:", len(my_user_factors))
print("User bias:", round(my_user_bias, 4))

Personalized user profile created!
Number of latent factors: 100
User bias: 0.5988


In [204]:
def predict_new_user_rating(model, user_factors, user_bias, movie_id):
    trainset = model.trainset

    movie_inner_id = get_movie_inner_id(trainset, movie_id)

    if movie_inner_id is None:
        return None

    movie_factors = model.qi[movie_inner_id]
    movie_bias = model.bi[movie_inner_id]

    predicted_rating = (
        trainset.global_mean
        + user_bias
        + movie_bias
        + np.dot(user_factors, movie_factors)
    )

    return float(np.clip(predicted_rating, 0.5, 5.0))

In [205]:
print("my_user_factors exists:", "my_user_factors" in globals())
print("my_user_bias exists:", "my_user_bias" in globals())

my_user_factors exists: True
my_user_bias exists: True


In [206]:
def get_movie_inner_id(trainset, movie_id):
    try:
        return trainset.to_inner_iid(movie_id)
    except ValueError:
        return None

In [207]:
rated_movie_ids = set(my_ratings.keys())

test_movie_ids = [10, 20, 100, 500, 1000]

for movie_id in test_movie_ids:
    if movie_id in rated_movie_ids:
        continue

    predicted_rating = predict_new_user_rating(
        svd_model,
        my_user_factors,
        my_user_bias,
        movie_id
    )

    movie_match = movies.loc[
        movies["movieId"] == movie_id,
        "title"
    ]

    movie_title = (
        movie_match.iloc[0]
        if not movie_match.empty
        else f"Movie ID {movie_id}"
    )

    if predicted_rating is None:
        print(f"{movie_title}: Skipped (not in training data)")
    else:
        print(f"{movie_title}: {predicted_rating:.2f}/5")

GoldenEye (1995): 4.09/5
Money Train (1995): 3.47/5
City Hall (1996): 3.54/5
Mrs. Doubtfire (1993): 4.03/5
Movie ID 1000: Skipped (not in training data)


In [208]:
def recommend_for_new_user(model, user_ratings, movies, top_n=10):
    if not user_ratings:
        raise ValueError("Please rate at least one movie.")

    # Estimate the new user's preferences
    user_factors, user_bias = estimate_new_user_preferences(
        model,
        user_ratings
    )

    # Exclude movies the user has already rated
    rated_movie_ids = set(user_ratings.keys())

    recommendations = []

    # Predict ratings for unrated movies
    for movie_id in movies["movieId"].unique():

        if movie_id in rated_movie_ids:
            continue

        predicted_rating = predict_new_user_rating(
            model,
            user_factors,
            user_bias,
            movie_id
        )

        if predicted_rating is None:
            continue

        movie_match = movies.loc[
            movies["movieId"] == movie_id
        ]

        if movie_match.empty:
            continue

        movie_info = movie_match.iloc[0]

        recommendations.append({
            "movieId": movie_id,
            "title": movie_info["title"],
            "genres": movie_info.get("genres", "Unknown"),
            "predicted_rating": predicted_rating
        })

    # Sort by predicted rating
    recommendations = sorted(
        recommendations,
        key=lambda x: x["predicted_rating"],
        reverse=True
    )

    return pd.DataFrame(recommendations[:top_n])

In [209]:
recommendations = recommend_for_new_user(
    model=svd_model,
    user_ratings=my_ratings,
    movies=movies,
    top_n=10
)

display(recommendations)

,movieId,title,genres,predicted_rating
0,1204,Lawrence of Arabia (1962),Adventure|Drama|War,4.970946
1,1178,Paths of Glory (1957),Drama|War,4.961648
2,1172,Cinema Paradiso (Nuovo cinema Paradiso) (1989),Drama,4.937764
3,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,4.937162
4,177593,"Three Billboards Outside Ebbing, Missouri (2017)",Crime|Drama,4.927707
5,318,"Shawshank Redemption, The (1994)",Crime|Drama,4.910014
6,1945,On the Waterfront (1954),Crime|Drama,4.907383
7,1104,"Streetcar Named Desire, A (1951)",Drama,4.906865
8,904,Rear Window (1954),Mystery|Thriller,4.903626
9,3451,Guess Who's Coming to Dinner (1967),Drama,4.894072


In [210]:
# Recommendations for a user who likes the selected movies
first_recommendations = recommend_for_new_user(
    model=svd_model,
    user_ratings=my_ratings,
    movies=movies,
    top_n=10
)
another_user_ratings = {
    1: 1.0,
    50: 1.5,
    260: 1.0
}

print("Second user's ratings created successfully!")

# Recommendations for a user who dislikes the same movies
second_recommendations = recommend_for_new_user(
    model=svd_model,
    user_ratings=another_user_ratings,
    movies=movies,
    top_n=10
)

# Compare the predicted ratings
print("First user:")
display(
    first_recommendations[
        ["title", "predicted_rating"]
    ]
)

print("Second user:")
display(
    second_recommendations[
        ["title", "predicted_rating"]
    ]
)

Second user's ratings created successfully!
First user:


,title,predicted_rating
0,Lawrence of Arabia (1962),4.970946
1,Paths of Glory (1957),4.961648
2,Cinema Paradiso (Nuovo cinema Paradiso) (1989),4.937764
3,Dr. Strangelove or: How I Learned to Stop Worr...,4.937162
4,"Three Billboards Outside Ebbing, Missouri (2017)",4.927707
5,"Shawshank Redemption, The (1994)",4.910014
6,On the Waterfront (1954),4.907383
7,"Streetcar Named Desire, A (1951)",4.906865
8,Rear Window (1954),4.903626
9,Guess Who's Coming to Dinner (1967),4.894072


Second user:


,title,predicted_rating
0,Paths of Glory (1957),2.113137
1,"Seventh Seal, The (Sjunde inseglet, Det) (1957)",2.103627
2,"Streetcar Named Desire, A (1951)",2.060677
3,"Sweet Hereafter, The (1997)",1.997916
4,Lawrence of Arabia (1962),1.995151
5,Double Indemnity (1944),1.983905
6,"Boondock Saints, The (2000)",1.960541
7,"Three Billboards Outside Ebbing, Missouri (2017)",1.959576
8,High Noon (1952),1.952441
9,Once Upon a Time in the West (C'era una volta ...,1.930185


In [211]:
print(recommendations.head(10).to_string(index=False))

 movieId                                                                       title              genres  predicted_rating
    1204                                                   Lawrence of Arabia (1962) Adventure|Drama|War          4.970946
    1178                                                       Paths of Glory (1957)           Drama|War          4.961648
    1172                              Cinema Paradiso (Nuovo cinema Paradiso) (1989)               Drama          4.937764
     750 Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)          Comedy|War          4.937162
  177593                            Three Billboards Outside Ebbing, Missouri (2017)         Crime|Drama          4.927707
     318                                            Shawshank Redemption, The (1994)         Crime|Drama          4.910014
    1945                                                    On the Waterfront (1954)         Crime|Drama          4.907383
    1104        

In [212]:
another_user_ratings = {
    1: 1.0,
    50: 1.5,
    260: 1.0
}

another_recommendations = recommend_for_new_user(
    model=svd_model,
    user_ratings=another_user_ratings,
    movies=movies,
    top_n=10
)

another_recommendations[
    ["title", "predicted_rating"]
]

,title,predicted_rating
0,Paths of Glory (1957),2.113137
1,"Seventh Seal, The (Sjunde inseglet, Det) (1957)",2.103627
2,"Streetcar Named Desire, A (1951)",2.060677
3,"Sweet Hereafter, The (1997)",1.997916
4,Lawrence of Arabia (1962),1.995151
5,Double Indemnity (1944),1.983905
6,"Boondock Saints, The (2000)",1.960541
7,"Three Billboards Outside Ebbing, Missouri (2017)",1.959576
8,High Noon (1952),1.952441
9,Once Upon a Time in the West (C'era una volta ...,1.930185


In [213]:
comparison = pd.DataFrame({
    "First User": recommendations["title"].head(10).reset_index(drop=True),
    "Second User": another_recommendations["title"].head(10).reset_index(drop=True)
})

comparison

,First User,Second User
0,Lawrence of Arabia (1962),Paths of Glory (1957)
1,Paths of Glory (1957),"Seventh Seal, The (Sjunde inseglet, Det) (1957)"
2,Cinema Paradiso (Nuovo cinema Paradiso) (1989),"Streetcar Named Desire, A (1951)"
3,Dr. Strangelove or: How I Learned to Stop Worr...,"Sweet Hereafter, The (1997)"
4,"Three Billboards Outside Ebbing, Missouri (2017)",Lawrence of Arabia (1962)
5,"Shawshank Redemption, The (1994)",Double Indemnity (1944)
6,On the Waterfront (1954),"Boondock Saints, The (2000)"
7,"Streetcar Named Desire, A (1951)","Three Billboards Outside Ebbing, Missouri (2017)"
8,Rear Window (1954),High Noon (1952)
9,Guess Who's Coming to Dinner (1967),Once Upon a Time in the West (C'era una volta ...


In [214]:
def get_movie_inner_id(trainset, movie_id):
    """Find a movie in the trained SVD model using either ID format."""

    for raw_id in (int(movie_id), str(int(movie_id))):
        try:
            return trainset.to_inner_iid(raw_id)
        except (ValueError, KeyError):
            continue

    return None


def estimate_new_user_profile(
    model,
    user_ratings,
    epochs=200,
    learning_rate=0.005,
    reg=0.1
):
    if not user_ratings:
        raise ValueError("Please rate at least one movie.")

    trainset = model.trainset

    user_factors = np.zeros(model.n_factors)
    user_bias = 0.0
    known_ratings = []

    for movie_id, actual_rating in user_ratings.items():

        inner_id = get_movie_inner_id(
            trainset,
            movie_id
        )

        if inner_id is not None:
            known_ratings.append(
                (inner_id, float(actual_rating))
            )

    if not known_ratings:
        raise ValueError(
            "None of your rated movies are available in "
            "the trained SVD model. Check the movie ID mapping."
        )

    for _ in range(epochs):
        for inner_id, actual_rating in known_ratings:

            movie_factors = model.qi[inner_id]
            movie_bias = model.bi[inner_id]

            predicted_rating = (
                trainset.global_mean
                + user_bias
                + movie_bias
                + np.dot(user_factors, movie_factors)
            )

            error = actual_rating - predicted_rating

            user_bias += learning_rate * (
                error - reg * user_bias
            )

            user_factors += learning_rate * (
                error * movie_factors
                - reg * user_factors
            )

    return user_factors, user_bias

In [215]:
my_ratings = {
    1: 5.0,
    50: 4.5,
    260: 5.0
}

my_user_factors, my_user_bias = estimate_new_user_profile(
    svd_model,
    my_ratings
)

print("New user profile created successfully!")
print("Number of latent factors:", len(my_user_factors))
print("Estimated user bias:", round(my_user_bias, 4))

New user profile created successfully!
Number of latent factors: 100
Estimated user bias: 0.5988


In [216]:


def predict_new_user_rating(
    model,
    user_factors,
    user_bias,
    movie_id
):
    """
    Predict a rating for a new user.
    Return None if the movie was not present during training.
    """

    trainset = model.trainset

    try:
        movie_inner_id = trainset.to_inner_iid(movie_id)
    except (ValueError, KeyError):
        try:
            movie_inner_id = trainset.to_inner_iid(str(movie_id))
        except (ValueError, KeyError):
            return None

    movie_factors = model.qi[movie_inner_id]
    movie_bias = model.bi[movie_inner_id]

    predicted_rating = (
        trainset.global_mean
        + user_bias
        + movie_bias
        + np.dot(user_factors, movie_factors)
    )

    return float(np.clip(predicted_rating, 0.5, 5.0))

In [217]:
test_movie_ids = [10, 20, 100, 500, 1000]

for movie_id in test_movie_ids:

    if movie_id in my_ratings:
        continue

    predicted_rating = predict_new_user_rating(
        svd_model,
        my_user_factors,
        my_user_bias,
        movie_id
    )

    movie_match = movies.loc[
        movies["movieId"] == movie_id,
        "title"
    ]

    movie_title = (
        movie_match.iloc[0]
        if not movie_match.empty
        else "Unknown movie"
    )

    if predicted_rating is None:
        print(f"{movie_title}: Cannot predict (movie not in training data)")
    else:
        print(f"{movie_title}: {predicted_rating:.2f}/5")

GoldenEye (1995): 4.09/5
Money Train (1995): 3.47/5
City Hall (1996): 3.54/5
Mrs. Doubtfire (1993): 4.03/5
Unknown movie: Cannot predict (movie not in training data)


In [218]:
print("Sample movie IDs in the model:")

print(
    list(svd_model.trainset._raw2inner_id_items.keys())[:10]
)

print("\nSample movie IDs in the dataset:")

print(
    movies["movieId"].head(10).tolist()
)

Sample movie IDs in the model:
[77866, 474, 4351, 2987, 1610, 177, 51662, 750, 6503, 8641]

Sample movie IDs in the dataset:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [219]:
# Step 1: Dataset validation

print("Movies dataset shape:", movies.shape)
print("Ratings dataset shape:", ratings.shape)

print("\nMissing values in ratings:")
print(ratings.isnull().sum())

print("\nDuplicate user-movie ratings:")
print(
    ratings.duplicated(
        subset=["userId", "movieId"]
    ).sum()
)

print("\nRating range:")
print("Minimum:", ratings["rating"].min())
print("Maximum:", ratings["rating"].max())

print("\nMissing movie IDs:")
print(movies["movieId"].isnull().sum())

print("\nDuplicate movie IDs:")
print(movies["movieId"].duplicated().sum())

print("\nRatings referencing nonexistent movies:")
print(
    len(
        set(ratings["movieId"]) -
        set(movies["movieId"])
    )
)

Movies dataset shape: (9742, 4)
Ratings dataset shape: (100836, 4)

Missing values in ratings:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Duplicate user-movie ratings:
0

Rating range:
Minimum: 0.5
Maximum: 5.0

Missing movie IDs:
0

Duplicate movie IDs:
0

Ratings referencing nonexistent movies:
0


In [220]:
print("Total ratings:", len(ratings))
print("Training ratings:", trainset.n_ratings)
print("Testing ratings:", len(testset))

print(
    "Training percentage:",
    round(trainset.n_ratings / len(ratings) * 100, 2),
    "%"
)

print(
    "Testing percentage:",
    round(len(testset) / len(ratings) * 100, 2),
    "%"
)

Total ratings: 100836
Training ratings: 80668
Testing ratings: 20168
Training percentage: 80.0 %
Testing percentage: 20.0 %


In [221]:
from surprise import accuracy

# Evaluate the current SVD model
svd_predictions = svd_model.test(testset)

# Calculate evaluation metrics
svd_rmse = accuracy.rmse(svd_predictions)
svd_mae = accuracy.mae(svd_predictions)

print("\nCurrent SVD Model Performance")
print("RMSE:", round(svd_rmse, 4))
print("MAE:", round(svd_mae, 4))


RMSE: 0.8718
MAE:  0.6700

Current SVD Model Performance
RMSE: 0.8718
MAE: 0.67
